# 17 — Small File Problem Pattern

Purpose: avoid creating too many tiny output files.

Process schema:

```text
MANY SMALL PARTITIONS
  |
  v
TOO MANY SMALL FILES
  |
  v
COALESCE / REPARTITION
  |
  v
REASONABLE FILE COUNT
```

Core idea:

```text
Number of Spark partitions often influences number of output files.
```

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *
import os
import shutil

spark = SparkSession.getActiveSession()
if spark is not None:
    spark.stop()

spark = (
    SparkSession.builder
    .appName("small-file-problem-pattern")
    .master("spark://spark-master:7077")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.warehouse.dir", "/tmp/spark_patterns_warehouse")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark.version

'4.0.2'

## Step 1 — Input data

In [2]:
rows = [
    (f"e{i}", f"u{i % 5}", float(i), "2026-01-01" if i < 50 else "2026-01-02")
    for i in range(100)
]

df = spark.createDataFrame(
    rows,
    ["event_id", "user_id", "amount", "event_date"]
)

df.show(10, truncate=False)
print("input partitions:", df.rdd.getNumPartitions())

[Stage 0:>                                                          (0 + 1) / 1]

+--------+-------+------+----------+
|event_id|user_id|amount|event_date|
+--------+-------+------+----------+
|e0      |u0     |0.0   |2026-01-01|
|e1      |u1     |1.0   |2026-01-01|
|e2      |u2     |2.0   |2026-01-01|
|e3      |u3     |3.0   |2026-01-01|
|e4      |u4     |4.0   |2026-01-01|
|e5      |u0     |5.0   |2026-01-01|
|e6      |u1     |6.0   |2026-01-01|
|e7      |u2     |7.0   |2026-01-01|
|e8      |u3     |8.0   |2026-01-01|
|e9      |u4     |9.0   |2026-01-01|
+--------+-------+------+----------+
only showing top 10 rows
input partitions: 2


## Step 2 — Force many partitions

This simulates a pipeline that creates too many output partitions.

In [3]:
many_partitions = df.repartition(20)

print("many_partitions:", many_partitions.rdd.getNumPartitions())

many_partitions: 20


## Step 3 — Write with too many partitions

Each partition can produce an output file.

In [4]:
base_path = "/tmp/spark_small_file_pattern"

too_many_path = f"{base_path}/too_many_files"
coalesce_path = f"{base_path}/coalesced"
partitioned_path = f"{base_path}/partitioned_by_date"

for path in [too_many_path, coalesce_path, partitioned_path]:
    shutil.rmtree(path, ignore_errors=True)

many_partitions.write.mode("overwrite").parquet(too_many_path)

part_files = [
    f for f in os.listdir(too_many_path)
    if f.startswith("part-") and f.endswith(".parquet")
]

print("parquet files:", len(part_files))

[Stage 4:=======================>                                  (8 + 2) / 20]

parquet files: 0


## Step 4 — Coalesce before write

Use `coalesce(n)` to reduce partitions without a full shuffle.

Good when:
- you only reduce partition count
- you do not need to redistribute by key

In [5]:
coalesced = many_partitions.coalesce(4)

print("coalesced partitions:", coalesced.rdd.getNumPartitions())

coalesced.write.mode("overwrite").parquet(coalesce_path)

part_files = [
    f for f in os.listdir(coalesce_path)
    if f.startswith("part-") and f.endswith(".parquet")
]

print("parquet files:", len(part_files))

coalesced partitions: 4
parquet files: 0


## Step 5 — Repartition by business column

Use `repartition(column)` when you want data redistributed by a key.

This causes a shuffle.

In [6]:
repartitioned_by_date = df.repartition("event_date")

print("repartitioned_by_date partitions:", repartitioned_by_date.rdd.getNumPartitions())

repartitioned_by_date.write.mode("overwrite").partitionBy("event_date").parquet(partitioned_path)

for root, dirs, files in os.walk(partitioned_path):
    parquet_files = [f for f in files if f.startswith("part-") and f.endswith(".parquet")]
    if parquet_files:
        print(root, "files:", len(parquet_files))

repartitioned_by_date partitions: 1


## Step 6 — Coalesce vs repartition

```text
coalesce(n)
- reduces number of partitions
- usually avoids full shuffle
- good before final write when data is already reasonably distributed

repartition(n)
- creates exactly/reasonably n partitions
- full shuffle
- good when data needs redistribution

repartition(column)
- redistributes data by column
- useful before partitioned writes or joins
```

In [7]:
comparison = spark.createDataFrame(
    [
        ("input", df.rdd.getNumPartitions()),
        ("many_partitions", many_partitions.rdd.getNumPartitions()),
        ("coalesced", coalesced.rdd.getNumPartitions()),
        ("repartitioned_by_date", repartitioned_by_date.rdd.getNumPartitions()),
    ],
    ["dataset", "partition_count"]
)

comparison.show(truncate=False)

+---------------------+---------------+
|dataset              |partition_count|
+---------------------+---------------+
|input                |2              |
|many_partitions      |20             |
|coalesced            |4              |
|repartitioned_by_date|1              |
+---------------------+---------------+



## Step 7 — Practical rule

```text
Do not blindly write every tiny partition as a file.
Control output partitions before write.
```

For large tables, target file size matters more than exact file count.
This notebook only demonstrates the mechanics.

In [8]:
spark.stop()